In [ ]:
import wrds
import pandas as pd
import yfinance as yf
import os

# ==========================================
# 1. Setup & WRDS Connection
# ==========================================
db = wrds.Connection(wrds_username="jlim356")

start_date = "2000-04-01"
end_date = "2026-04-01"

print("Fetching historical S&P 500 membership from CRSP...")

# Step 1: Historical S&P 500 constituent membership
# crsp.msp500list gives the permno + date range each stock was in the index
sql_sp500_members = f"""
SELECT permno, start, ending
FROM crsp.msp500list
WHERE start <= '{end_date}'
  AND (ending >= '{start_date}' OR ending IS NULL)
ORDER BY permno, start
"""
df_members = db.raw_sql(sql_sp500_members)
print(f"  Unique PERMNOs ever in S&P 500: {df_members['permno'].nunique()}")

# Step 2: Pull daily returns — only when each stock was IN the index
# ret = total return (price appreciation + dividends), adjusted by CRSP
# prc is negative in CRSP when it's a bid-ask midpoint, not a traded price
print("Fetching daily returns from CRSP DSF (this may take a few minutes)...")
sql_crsp_daily = f"""
SELECT
    a.permno,
    a.date,
    a.ret,
    ABS(a.prc) AS prc,
    a.shrout,
    a.vol
FROM crsp.dsf a
INNER JOIN crsp.msp500list b
    ON a.permno = b.permno
    AND a.date >= b.start
    AND a.date <= COALESCE(b.ending, CURRENT_DATE)
WHERE a.date >= '{start_date}'
  AND a.date <= '{end_date}'
ORDER BY a.date, a.permno
"""
df_crsp = db.raw_sql(sql_crsp_daily, date_cols=["date"])
print(f"  Raw rows fetched: {len(df_crsp):,}")

# Step 3: Ticker/name mapping for interpretability
# msenames maps permno -> ticker symbol, using the most recent name per permno
sql_names = """
SELECT DISTINCT permno, ticker, comnam, namedt, nameendt
FROM crsp.msenames
WHERE ticker IS NOT NULL
ORDER BY permno, namedt
"""
df_names = db.raw_sql(sql_names)
df_names_latest = df_names.sort_values("namedt").groupby("permno").last().reset_index()[["permno", "ticker", "comnam"]]

# Step 4: Build wide returns matrix (T x N) for RMT analysis
# Shape: trading days (rows) x stocks (columns)
# NaNs occur when a stock was not yet / no longer in the index on that date
df_crsp["date"] = pd.to_datetime(df_crsp["date"]).dt.normalize()

df_returns_matrix = df_crsp.pivot_table(index="date", columns="permno", values="ret", aggfunc="first")

# Rename permno integer columns -> ticker symbols
permno_to_ticker = df_names_latest.set_index("permno")["ticker"].to_dict()
df_returns_matrix.columns.name = None
df_returns_matrix = df_returns_matrix.rename(columns=permno_to_ticker)

print(f"\nReturns matrix shape: {df_returns_matrix.shape}")
print(f"  Date range: {df_returns_matrix.index.min().date()} to {df_returns_matrix.index.max().date()}")
print(f"  Unique stocks (ever-constituent): {df_returns_matrix.shape[1]}")
print(f"  Overall NaN density: {df_returns_matrix.isna().mean().mean():.1%}")
print(f"  Avg non-NaN stocks per day: {df_returns_matrix.notna().sum(axis=1).mean():.0f}")

# Step 5: Save outputs
output_dir = "data_raw"
os.makedirs(output_dir, exist_ok=True)

df_returns_matrix.to_parquet(f"{output_dir}/sp500_returns_matrix.parquet", engine="pyarrow", compression="snappy")

# Long-form with price/volume (useful for market-cap weighting later)
df_crsp_long = df_crsp.merge(df_names_latest, on="permno", how="left")
df_crsp_long.to_parquet(f"{output_dir}/sp500_crsp_long.parquet", engine="pyarrow", compression="snappy")

# Ticker-to-company name lookup (for labeling eigenvectors in RMT analysis)
df_names_latest.to_parquet(f"{output_dir}/permno_ticker_map.parquet", engine="pyarrow", compression="snappy")

print(f"\nSaved: sp500_returns_matrix.parquet  — shape {df_returns_matrix.shape}")
print(f"Saved: sp500_crsp_long.parquet       — {len(df_crsp_long):,} rows")
print(f"Saved: permno_ticker_map.parquet     — {len(df_names_latest)} stocks")


# Export to Parquet
db.close()

In [ ]:

print("Fetching historical S&P 500 membership from CRSP...")

# Step 1: Historical S&P 500 constituent membership
# crsp.msp500list gives the permno + date range each stock was in the index
sql_sp500_members = f"""
SELECT permno, start, ending
FROM crsp.msp500list
WHERE start <= '{end_date}'
  AND (ending >= '{start_date}' OR ending IS NULL)
ORDER BY permno, start
"""
df_members = db.raw_sql(sql_sp500_members)
print(f"  Unique PERMNOs ever in S&P 500: {df_members['permno'].nunique()}")

# Step 2: Pull daily returns — only when each stock was IN the index
# ret = total return (price appreciation + dividends), adjusted by CRSP
# prc is negative in CRSP when it's a bid-ask midpoint, not a traded price
print("Fetching daily returns from CRSP DSF (this may take a few minutes)...")
sql_crsp_daily = f"""
SELECT
    a.permno,
    a.date,
    a.ret,
    ABS(a.prc) AS prc,
    a.shrout,
    a.vol
FROM crsp.dsf a
INNER JOIN crsp.msp500list b
    ON a.permno = b.permno
    AND a.date >= b.start
    AND a.date <= COALESCE(b.ending, CURRENT_DATE)
WHERE a.date >= '{start_date}'
  AND a.date <= '{end_date}'
ORDER BY a.date, a.permno
"""
df_crsp = db.raw_sql(sql_crsp_daily, date_cols=["date"])
print(f"  Raw rows fetched: {len(df_crsp):,}")

# Step 3: Ticker/name mapping for interpretability
# msenames maps permno -> ticker symbol, using the most recent name per permno
sql_names = """
SELECT DISTINCT permno, ticker, comnam, namedt, nameendt
FROM crsp.msenames
WHERE ticker IS NOT NULL
ORDER BY permno, namedt
"""
df_names = db.raw_sql(sql_names)
df_names_latest = (
    df_names
    .sort_values("namedt")
    .groupby("permno")
    .last()
    .reset_index()[["permno", "ticker", "comnam"]]
)

# Step 4: Build wide returns matrix (T x N) for RMT analysis
# Shape: trading days (rows) x stocks (columns)
# NaNs occur when a stock was not yet / no longer in the index on that date
df_crsp["date"] = pd.to_datetime(df_crsp["date"]).dt.normalize()

df_returns_matrix = df_crsp.pivot_table(
    index="date",
    columns="permno",
    values="ret",
    aggfunc="first"
)

# Rename permno integer columns -> ticker symbols
permno_to_ticker = df_names_latest.set_index("permno")["ticker"].to_dict()
df_returns_matrix.columns.name = None
df_returns_matrix = df_returns_matrix.rename(columns=permno_to_ticker)

print(f"\nReturns matrix shape: {df_returns_matrix.shape}")
print(f"  Date range: {df_returns_matrix.index.min().date()} to {df_returns_matrix.index.max().date()}")
print(f"  Unique stocks (ever-constituent): {df_returns_matrix.shape[1]}")
print(f"  Overall NaN density: {df_returns_matrix.isna().mean().mean():.1%}")
print(f"  Avg non-NaN stocks per day: {df_returns_matrix.notna().sum(axis=1).mean():.0f}")

# Step 5: Save outputs
df_returns_matrix.to_parquet(
    f"{output_dir}/sp500_returns_matrix.parquet",
    engine="pyarrow",
    compression="snappy"
)

# Long-form with price/volume (useful for market-cap weighting later)
df_crsp_long = df_crsp.merge(df_names_latest, on="permno", how="left")
df_crsp_long.to_parquet(
    f"{output_dir}/sp500_crsp_long.parquet",
    engine="pyarrow",
    compression="snappy"
)

# Ticker-to-company name lookup (for labeling eigenvectors in RMT analysis)
df_names_latest.to_parquet(
    f"{output_dir}/permno_ticker_map.parquet",
    engine="pyarrow",
    compression="snappy"
)

print(f"\nSaved: sp500_returns_matrix.parquet  — shape {df_returns_matrix.shape}")
print(f"Saved: sp500_crsp_long.parquet       — {len(df_crsp_long):,} rows")
print(f"Saved: permno_ticker_map.parquet     — {len(df_names_latest)} stocks")